This Python code will only work if your ESP32 is configured to act as a Bluetooth client, connecting to your computer's Bluetooth server (the Python script). The ESP32 firmware (written in C++ with the Arduino framework or MicroPython) must:

Use the Bluetooth Serial Port Profile (SPP) to establish a classic Bluetooth connection.

Read raw audio data from an I2S microphone.

Stream that raw audio data to the connected client (your computer) over the Bluetooth connection.

In [ ]:
**PyBluez is difficult to install on Windows. You might need to install additional system-level Bluetooth libraries first:
  Microsoft Visual C++ Build Tools
  Windows SDK

The most common and reliable method involves setting up the ESP32 as a Bluetooth A2DP source and a PC as a Bluetooth A2DP sink :
or using a custom profile for data transfer. 


In [ ]:
Bluetooth audio streaming typically uses the Advanced Audio Distribution Profile (A2DP), 
which is designed for devices like speakers and headphones to receive audio, not for a computer to act as a receiver 
for raw microphone data. You'll need to use either the Serial Port Profile (SPP) or a custom Bluetooth profile to 
transfer raw data from the ESP32 to your computer.

In [2]:
pip install pybluez

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  python setup.py egg_info did not run successfully.
  exit code: 1
  
  [1 lines of output]
  error in PyBluez setup command: use_2to3 is invalid.
  [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
error: metadata-generation-failed

Encountered error while generating package metadata.

See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [3]:
import bluetooth
import pyaudio
import wave
import sys

ModuleNotFoundError: No module named 'bluetooth'

In [ ]:
# --- Bluetooth Configuration ---
# Find this on your ESP32's firmware or through a Bluetooth scanner app
# The MAC address of your ESP32 device
ESP32_MAC_ADDRESS = "XX:XX:XX:XX:XX:XX" 
# This should match the channel used on the ESP32's firmware (e.g., SPP profile)
PORT = 1 

# --- Audio Configuration ---
FORMAT = pyaudio.paInt16
CHANNELS = 1 
RATE = 16000 
CHUNK = 1024
WAVE_OUTPUT_FILENAME = "bird_recording.wav"

In [ ]:
def receive_and_save_audio():
    # --- Initialize PyAudio ---
    audio = pyaudio.PyAudio()

    # --- Create a Bluetooth Socket Server ---
    server_sock = bluetooth.BluetoothSocket(bluetooth.RFCOMM)
    server_sock.bind(("", PORT))
    server_sock.listen(1)
    
    print(f"Waiting for connection on RFCOMM channel {PORT}...")
    
    try:
        client_sock, client_info = server_sock.accept()
        print(f"Accepted connection from {client_info}")

        # --- Open WAV file to save the data ---
        wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(audio.get_sample_size(FORMAT))
        wf.setframerate(RATE)

        print("Receiving audio stream...")
        while True:
            # Receive data from the ESP32
            data = client_sock.recv(CHUNK * CHANNELS * 2) # 2 bytes per sample for paInt16
            if not data:
                break
            # Write the received audio data to the WAV file
            wf.writeframes(data)
            
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        print("Closing connections.")
        client_sock.close()
        server_sock.close()
        wf.close()
        audio.terminate()
        print(f"Audio capture stopped. Saved to {WAVE_OUTPUT_FILENAME}")

if __name__ == "__main__":
    receive_and_save_audio()